# 07. Palindrome Patterns and Manacher's Algorithm

A palindrome reads the same in both directions. Palindrome questions are very common, and there is a **ladder of solutions** from simple to optimal.

**Learning order:** LADDER → CHECK → EXPAND AROUND CENTER → MANACHER → COUNT → OTHER PALINDROME TASKS → INTERVIEW THINKING

## 1. The ladder (know all rungs)

| Task | Approach | Time |
|---|---|---|
| is `s` a palindrome | two pointers from both ends | O(n) |
| longest palindromic substring | **expand around centers** | O(n²), O(1) space |
| | DP table `is_pal[i][j]` | O(n²), O(n²) space |
| | **Manacher** | **O(n)** |
| longest palindromic subsequence | DP / LCS with reverse | O(n²) (DP topic) |
| partition into palindromes | backtracking / DP (Backtracking, DP topics) | |
| shortest palindrome by adding in front | KMP / rolling hash (next notebook) | O(n) |

In [1]:
def is_palindrome(s):
    l, r = 0, len(s) - 1
    while l < r:
        if s[l] != s[r]:
            return False
        l += 1
        r -= 1
    return True

def expand_longest(s):                                   # O(n^2), O(1) space
    best = ""
    for center in range(2 * len(s) - 1):                 # 2n - 1 centers: letters and gaps
        l, r = center // 2, center // 2 + center % 2
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l -= 1
            r += 1
        if r - l - 1 > len(best):
            best = s[l + 1:r]
    return best

print(is_palindrome("level"), is_palindrome("abca"), expand_longest("babad"), expand_longest("cbbd"))   # True False bab bb

True False bab bb


## 2. Manacher's algorithm (O(n))

Expanding from every center repeats work. Manacher **reuses** information from a palindrome already found.

**Step 1: make all palindromes odd** by inserting `#` between characters (`"abba"` → `"#a#b#b#a#"`). Now every palindrome has a single center.

**Step 2:** `p[i]` = radius of the palindrome centered at `i` in the new string. Keep the **rightmost palindrome** `(center c, right edge r)`.
- If `i < r`, its mirror position is `2c - i`, so start with `p[i] = min(r - i, p[mirror])` (we already know this much is a palindrome).
- Then try to **expand** further. If it passes `r`, update `c` and `r`.

Each expansion step moves `r` to the right and `r` never moves back, so the total work is **O(n)**.

In [2]:
def manacher(s):
    t = "#" + "#".join(s) + "#"
    n = len(t)
    p = [0] * n
    c = r = 0                                              # center and right edge of the rightmost palindrome
    for i in range(n):
        if i < r:
            p[i] = min(r - i, p[2 * c - i])                # reuse the mirror's radius
        while i - p[i] - 1 >= 0 and i + p[i] + 1 < n and t[i - p[i] - 1] == t[i + p[i] + 1]:
            p[i] += 1                                      # expand
        if i + p[i] > r:
            c, r = i, i + p[i]
    return p

def longest_palindrome_manacher(s):
    p = manacher(s)
    i = max(range(len(p)), key=lambda k: p[k])             # center of the biggest palindrome
    start = (i - p[i]) // 2
    return s[start:start + p[i]]

print(longest_palindrome_manacher("babad"), longest_palindrome_manacher("cbbd"), longest_palindrome_manacher("forgeeksskeegfor"))   # bab bb geeksskeeg

bab bb geeksskeeg


## 3. Counting all palindromic substrings

In the transformed string a center with radius `p[i]` contributes `(p[i] + 1) // 2` palindromic substrings. Total in O(n).

In [3]:
def count_palindromes(s):
    return sum((r + 1) // 2 for r in manacher(s))

print(count_palindromes("abc"), count_palindromes("aaa"))   # 3 6

3 6


## 4. Other palindrome tasks

- **Valid Palindrome II** (delete at most one character): on the first mismatch try skipping the left or the right character (Two Pointers topic).
- **Palindrome Permutation:** a string can be rearranged into a palindrome iff **at most one character has an odd count**.
- **Longest palindrome that can be built from letters:** use all pairs, plus one odd leftover (Hashing topic).
- **Minimum insertions to make a palindrome** = `n - longest palindromic subsequence` (DP topic).

In [4]:
from collections import Counter

def can_permute_palindrome(s):
    return sum(c % 2 for c in Counter(s).values()) <= 1

print(can_permute_palindrome("carerac"), can_permute_palindrome("code"))   # True False

True False


## 5. Interview thinking

**Say:** "Checking a palindrome is O(n). For the longest palindromic substring I expand around each of the 2n-1 centers: O(n²) time, O(1) space. Manacher brings it to O(n) by reusing mirrored radii."

**In most interviews, expand-around-center is the expected answer.** Mention Manacher as the optimization.

**Common mistakes:** forgetting **even-length** palindromes (gap centers), off-by-one when slicing the result, `r - l - 1` length formula, using DP O(n²) memory when O(1) is possible.

## Quick Revision

- Expand around 2n-1 centers: O(n²), O(1) space.
- Manacher: insert `#`, keep rightmost `(c, r)`, `p[i] = min(r-i, p[mirror])`, expand.
- Palindrome permutation ⇔ at most one odd count.
- Don't forget even-length palindromes.